# Training Ulang Model dari Hyperparameter Tersimpan

Melanjutkan `code_run_all.ipynb` tanpa menjalankan Genetic Algorithm lagi: hyperparameter dibaca dari `params/best_params_lgbm.pkl` dan `params/best_params_xgb.pkl`, lalu dipakai melatih ulang, mengevaluasi, dan menyimpan model. Sel pipeline di bawah salinan identik dari `code_run_all.ipynb`, jadi split train/test dan konfigurasi fitur sama persis dengan saat hyperparameter itu dicari.

## Import Library

In [ ]:
import warnings
from pathlib import Path

import joblib as jl
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from skforecast.direct import ForecasterDirect
from skforecast.exceptions import IgnoredArgumentWarning, LongTrainingWarning
from skforecast.model_selection import TimeSeriesFold, backtesting_forecaster
from skforecast.preprocessing import CalendarFeatures, RollingFeatures
from skforecast.utils import save_forecaster
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    root_mean_squared_error,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import FunctionTransformer, StandardScaler

In [ ]:
warnings.simplefilter("ignore", category=IgnoredArgumentWarning)
warnings.simplefilter("ignore", category=LongTrainingWarning)

## Memuat Dataset & Split Train/Test

In [ ]:
df_raw = pd.read_csv("DATASET-BERAS.csv")
df_raw.head()
df_raw.index = pd.DatetimeIndex(df_raw["tanggal"], yearfirst=True)
df_raw.drop(columns=["tanggal"], inplace=True)
df_raw = df_raw.asfreq("D")
holdout = TimeSeriesSplit(
    n_splits=5,
)

train_idx, test_idx = list(holdout.split(df_raw))[-1]
raw_train = df_raw.iloc[train_idx].copy()
raw_test = df_raw.iloc[test_idx].copy()

train_filled_interpolation = raw_train.interpolate(
    method="time", limit_direction="both"
)
test_filled_interpolation = (
    pd.concat([train_filled_interpolation.tail(1), raw_test]).ffill().iloc[1:]
)
df_cleaned = pd.concat([train_filled_interpolation, test_filled_interpolation]).loc[
    raw_train.index[0] :
]

assert df_cleaned.index.equals(df_raw.loc[raw_train.index[0] :].index)
assert not df_cleaned.isna().any().any()
print("Train:", raw_train.index.min().date(), "->", raw_train.index.max().date())
print("Test:", raw_test.index.min().date(), "->", raw_test.index.max().date())

## Transformasi Log

In [ ]:
commodity_cols = df_cleaned.columns.tolist()

log_transformation = FunctionTransformer(
    func=np.log1p, inverse_func=np.expm1, check_inverse=True
)

df_cleaned_log = pd.DataFrame(
    log_transformation.fit_transform(df_cleaned[commodity_cols]),
    index=df_cleaned.index,
    columns=commodity_cols,
)

def log_to_price(log_values):
    """Kembalikan prediksi skala log1p ke skala harga."""
    return np.expm1(np.asarray(log_values, dtype=float).ravel())


data_by_col = {}

for col in commodity_cols:
    data_by_col[col] = {
        "price_train": df_cleaned.loc[raw_train.index, col],
        "price_test": df_cleaned.loc[raw_test.index, col],
        "log_price_train": df_cleaned_log.loc[raw_train.index, col].rename(col),
        "log_price_test": df_cleaned_log.loc[raw_test.index, col].rename(col),
    }

## Menyiapkan Fitur dan Forecaster

In [ ]:
horizons = [1, 7, 30]
LAGS = [1, 7, 30]
DIFFERENTIATION = 1  # model memprediksi perubahan harian log harga

window_features = RollingFeatures(
    stats=["mean", "mean", "std", "std", "min", "min", "max", "max"],
    window_sizes=[7, 30, 7, 30, 7, 30, 7, 30],
)
calendar_features = CalendarFeatures(
    features=["day_of_year", "day_of_week", "day_of_month", "month"],
    features_to_encode=["day_of_year", "day_of_month", "day_of_week", "month"],
    encoding="cyclical",
    max_values={
        "day_of_year": 365,
        "day_of_week": 6,
        "day_of_month": 31,
        "month": 12,
    },
)


def make_forecaster(estimator, steps):
    return ForecasterDirect(
        estimator=estimator,
        steps=steps,
        lags=LAGS,
        window_features=window_features,
        calendar_features=calendar_features,
        transformer_y=StandardScaler(),
        differentiation=DIFFERENTIATION,
    )


def prediction_column(predictions):
    return "pred" if "pred" in predictions.columns else predictions.columns[0]

## Memuat Hyperparameter Tersimpan

In [ ]:
MODEL_SPECS = {
    "LightGBM": {
        "params_path": "params/best_params_lgbm.pkl",
        "estimator": lgb.LGBMRegressor,
        "folder": "final_lgbm",
        "prefix": "final_lgbm",
    },
    "XGBoost": {
        "params_path": "params/best_params_xgb.pkl",
        "estimator": xgb.XGBRegressor,
        "folder": "final_xgb",
        "prefix": "final_xgb",
    },
}

best_params = {}

for model_name, spec in MODEL_SPECS.items():
    params_path = Path(spec["params_path"])
    if params_path.exists():
        best_params[model_name] = jl.load(params_path)
        print(f"{model_name}: parameter dimuat dari {params_path}")
    else:
        print(f"{model_name}: {params_path} belum ada — dilewati")

assert best_params, "Tidak ada file parameter. Jalankan pencarian di code_run_all.ipynb dulu."


tidak_lengkap = [
    (model_name, col, h)
    for model_name, params_store in best_params.items()
    for col in commodity_cols
    for h in horizons
    if h not in params_store.get(col, {})
]
assert not tidak_lengkap, f"Parameter tidak lengkap untuk: {tidak_lengkap[:5]}"
print(f"\nModel aktif: {', '.join(best_params)}")

## Melatih Ulang Forecaster

In [ ]:
forecasters = {}

for model_name, params_store in best_params.items():
    estimator_cls = MODEL_SPECS[model_name]["estimator"]
    forecasters[model_name] = {}

    for col in commodity_cols:
        forecasters[model_name][col] = {}
        for h in horizons:
            forecaster = make_forecaster(
                estimator_cls(**params_store[col][h]["params"]), h
            )
            forecaster.fit(y=data_by_col[col]["log_price_train"])
            forecasters[model_name][col][h] = forecaster

    print(
        f"{model_name}: {len(commodity_cols) * len(horizons)} forecaster terlatih "
        f"({len(commodity_cols)} komoditas × {len(horizons)} horizon)"
    )

## Evaluasi pada Data Test

In [ ]:
eval_rows = []

for model_name, model_forecasters in forecasters.items():
    for col in commodity_cols:
        price_test = data_by_col[col]["price_test"]
        for h in horizons:
            price_pred = log_to_price(model_forecasters[col][h].predict(steps=h))
            actual_price = price_test.iloc[:h].to_numpy()
            eval_rows.append(
                {
                    "Model": model_name,
                    "Komoditas": col,
                    "Horizon": h,
                    "MAE": mean_absolute_error(actual_price, price_pred),
                    "RMSE": root_mean_squared_error(actual_price, price_pred),
                    "MAPE (%)": mean_absolute_percentage_error(
                        actual_price, price_pred
                    )
                    * 100,
                }
            )

df_eval = pd.DataFrame(eval_rows)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
df_eval

## Ringkasan MAPE Antar Model

In [ ]:
df_mape_pivot = df_eval.pivot_table(
    index="Komoditas", columns=["Model", "Horizon"], values="MAPE (%)"
)
if len(df_eval["Model"].unique()) > 1:
    df_mape_pivot["Model Terbaik"] = (
        df_eval.pivot_table(index="Komoditas", columns="Model", values="MAPE (%)")
        .idxmin(axis=1)
    )
df_mape_pivot

## Backtesting Walk-forward

In [ ]:
wf_rows = []
prediksi_per_model = {}

for model_name, params_store in best_params.items():
    estimator_cls = MODEL_SPECS[model_name]["estimator"]
    plot_rows = []

    for col in commodity_cols:
        log_price_train = data_by_col[col]["log_price_train"]
        y_walk_forward = pd.concat(
            [log_price_train, data_by_col[col]["log_price_test"]]
        ).asfreq("D")

        for h in horizons:
            cv = TimeSeriesFold(
                steps=h,
                initial_train_size=len(log_price_train),
                refit=True,
                fixed_train_size=False,
                allow_incomplete_fold=True,
                differentiation=DIFFERENTIATION,
                verbose=False,
            )
            _, predictions = backtesting_forecaster(
                forecaster=make_forecaster(
                    estimator_cls(**params_store[col][h]["params"]), h
                ),
                y=y_walk_forward,
                cv=cv,
                metric="mean_absolute_error",
                verbose=False,
                show_progress=False,
            )

            idx = predictions.index
            price_predicted = log_to_price(predictions[prediction_column(predictions)])
            raw_actual = raw_test[col].reindex(idx)
            real_mask = raw_actual.notna().to_numpy()
            price_actual = raw_actual.to_numpy()[real_mask]
            price_predicted = price_predicted[real_mask]

            wf_rows.append(
                {
                    "Model": model_name,
                    "Komoditas": col,
                    "Horizon": h,
                    "Jumlah Observasi": int(real_mask.sum()),
                    "MAE": mean_absolute_error(price_actual, price_predicted),
                    "RMSE": root_mean_squared_error(price_actual, price_predicted),
                    "MAPE (%)": mean_absolute_percentage_error(
                        price_actual, price_predicted
                    )
                    * 100,
                }
            )
            plot_rows.extend(
                {
                    "Tanggal": tanggal,
                    "Komoditas": col,
                    "Horizon": h,
                    "Harga Aktual": actual,
                    "Harga Prediksi": predicted,
                }
                for tanggal, actual, predicted in zip(
                    idx[real_mask], price_actual, price_predicted
                )
            )

    prediksi_per_model[model_name] = pd.DataFrame(plot_rows)
    print(f"{model_name}: walk-forward selesai")

df_walkforward = pd.DataFrame(wf_rows)
df_walkforward

## Plot Harga Aktual vs Prediksi Walk-forward

In [ ]:
WARNA_MODEL = {"LightGBM": "#d97941", "XGBoost": "#2b5c8f"}


def plot_walkforward(prediksi_per_model, judul_utama, lebar=16, tinggi_panel=2.9):
    """Satu panel lebar per (horizon x model): harga aktual dan prediksi
    ditumpuk dalam panel yang sama, panel-panelnya berurutan ke bawah.
    Satu figure per komoditas, bukan grid kotak kecil, supaya sumbu tanggal
    dan selisih harga masih terbaca saat dicetak.
    """
    kombinasi = [(h, nama) for h in horizons for nama in prediksi_per_model]

    for col in commodity_cols:
        fig, axes = plt.subplots(
            nrows=len(kombinasi),
            ncols=1,
            figsize=(lebar, tinggi_panel * len(kombinasi)),
            sharex=True,
            squeeze=False,
        )

        for row_idx, (h, nama_model) in enumerate(kombinasi):
            ax = axes[row_idx, 0]
            df_pred = prediksi_per_model[nama_model]
            panel = df_pred.loc[
                (df_pred["Komoditas"] == col) & (df_pred["Horizon"] == h)
            ].sort_values("Tanggal")

            if panel.empty:
                ax.text(0.5, 0.5, "Tidak ada observasi", ha="center", va="center")
                ax.set_axis_off()
                continue

            ax.plot(
                panel["Tanggal"],
                panel["Harga Aktual"],
                color="#355c7d",
                linewidth=2.0,
                label="Harga aktual",
            )
            ax.plot(
                panel["Tanggal"],
                panel["Harga Prediksi"],
                color=WARNA_MODEL.get(nama_model, "#8c564b"),
                linewidth=1.8,
                linestyle="--",
                label=f"Prediksi {nama_model}",
            )

            mape_panel = (
                abs(panel["Harga Prediksi"] - panel["Harga Aktual"])
                / panel["Harga Aktual"]
            ).mean() * 100
            ax.set_title(
                f"{nama_model} | H+{h} | MAPE {mape_panel:.2f}%",
                fontsize=11,
                loc="left",
            )
            ax.set_ylabel("Harga (Rp)", fontsize=10)
            ax.grid(axis="y", alpha=0.25)
            ax.tick_params(axis="both", labelsize=9)
            ax.legend(loc="upper left", fontsize=9, frameon=False, ncol=2)

        axes[-1, 0].set_xlabel("Tanggal", fontsize=10)
        axes[-1, 0].tick_params(axis="x", labelrotation=20, labelsize=9)
        fig.suptitle(f"{judul_utama} — {col}", fontsize=14, y=0.995)
        fig.tight_layout(rect=(0, 0, 1, 0.985))
        plt.show()

plot_walkforward(prediksi_per_model, "Walk-forward (parameter tersimpan)")

## Early Warning System H+30

In [ ]:
THRESHOLD = {"Bahaya": 20, "Warning": 10, "Waspada": 5}  # urut dari paling berat
h_ews = 30


def ews_status(persen):
    for status, batas in THRESHOLD.items():
        if persen > batas:
            return status
    return "Normal"


ews_rows = []

for model_name, model_forecasters in forecasters.items():
    for col in commodity_cols:
        price_train = data_by_col[col]["price_train"]
        harga_origin = float(price_train.iloc[-1])
        harga_pred = log_to_price(model_forecasters[col][h_ews].predict(steps=h_ews))
        harga_h30 = float(harga_pred[-1])
        persen = (harga_h30 - harga_origin) / harga_origin * 100

        ews_rows.append(
            {
                "Model": model_name,
                "Komoditas": col,
                "Tanggal origin": price_train.index[-1].date(),
                "Harga origin (train)": harga_origin,
                "Prediksi H+30": harga_h30,
                "Perubahan (%)": persen,
                "Status": ews_status(persen),
            }
        )

df_ews = pd.DataFrame(ews_rows)
df_ews

## Menyimpan Final Model

In [ ]:
def save_forecasters(forecasters, folder, prefix):
    Path(folder).mkdir(exist_ok=True)
    for col in commodity_cols:
        for h in horizons:
            save_forecaster(
                forecasters[col][h],
                str(Path(folder) / f"{prefix}_{col}_h{h}.joblib"),
                save_custom_functions=False,
            )
            print("tersimpan:", col, "h=" + str(h))

for model_name, model_forecasters in forecasters.items():
    spec = MODEL_SPECS[model_name]
    print(f"\n== {model_name} -> {spec['folder']} ==")
    save_forecasters(model_forecasters, spec["folder"], spec["prefix"])